In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Creating subagents

In [8]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x ** 2

In [3]:
from langchain.agents import create_agent

# create subagents

subagent_1 = create_agent(
    model='ollama:llama3.1:8b',
    tools=[square_root]
)

subagent_2 = create_agent(
    model='ollama:llama3.1:8b',
    tools=[square]
)

## Calling subagents

In [5]:
from langchain.messages import HumanMessage

@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = subagent_1.invoke({"messages": [HumanMessage(content=f"Calculate the square root of {x}")]})
    return response["messages"][-1].content

@tool
def call_subagent_2(x: float) -> float:
    """Call subagent 2 in order to calculate the square of a number"""
    response = subagent_2.invoke({"messages": [HumanMessage(content=f"Calculate the square of {x}")]})
    return response["messages"][-1].content

## Creating the main agent

main_agent = create_agent(
    model='ollama:llama3.1:8b',
    tools=[call_subagent_1, call_subagent_2],
    system_prompt="You are a helpful assistant who can call subagents to calculate the square root or square of a number.")

## Test

In [6]:
question = "What is the square root of 456?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

In [7]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is the square root of 456?', additional_kwargs={}, response_metadata={}, id='d2058a65-1d32-4d80-ae49-e3f18f9e72b4'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-09-03T10:14:54.2970806Z', 'done': True, 'done_reason': 'stop', 'total_duration': 11741435500, 'load_duration': 8496794500, 'prompt_eval_count': 245, 'prompt_eval_duration': 626508000, 'eval_count': 20, 'eval_duration': 2606524000, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'}, id='lc_run--01a066c3-7c79-7bf1-88c2-f58bcd83e6c9-0', tool_calls=[{'name': 'call_subagent_1', 'args': {'x': 456}, 'id': '964bdc8e-ad3e-430a-a288-f3e741766cb8', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 245, 'output_tokens': 20, 'total_tokens': 265}),
              ToolMessage(content='The square root of 456.0 is approximately 21.354156504062622.', name='call_subagent_1', id='2